# Notebook 05: EM capture pipeline walkthrough

This notebook is the entry point of the visual demonstrator
(notebooks `05`-`07`). It walks through a single EM capture from
the picoc target: how the raw traces look, how baseline
(calibration-phase) traces compare to operation-phase traces, and
where the reference figures emitted by the production capture
pipeline appear in the workflow.

## Capture setup

- Target software: picoc, a small open-source C interpreter.
- Hardware: an ARMv7 single-board computer.
- EM probe: a passive near-field H-field loop positioned a few
  millimetres above the SoC package.
- Acquisition: 10 GHz sample rate, 500 002 samples per trace
  (~50 us of capture time per execution).
- Calibration phase: 100 baseline traces produced by three
  microbenchmarks (compute-bound tight loop, memory-bound
  allocator, I/O-bound file-descriptor reads/writes). The
  resulting normalised feature vectors define the reference
  centroids consumed by the paper's EM hypothesis rule.
- Operation phase: 3 464 traces captured during a fuzzing
  campaign that exercises an integer-overflow trigger in picoc.

## Scope and disclaimers

Two scope statements matter for reading this notebook against
the paper:

1. The picoc capture used here is from a fuzzing campaign with
   a different vulnerability target than the picoc campaign in
   the paper. Per-trace counts and crash density therefore do
   not match the paper.
2. The `is_anomalous` column in `data/picoc_features.csv` is
   derived from `exit_code` (`True` when the picoc process
   exited abnormally). This is not the paper's notion of an
   EM-anomalous input, which is flagged by EM deviation from the
   calibration baseline before any crash classification. The
   demonstrator CSV labels every operation-phase trace by exit
   code, so its anomaly proportion reflects the crash rate of
   the fuzzing campaign, not an EM-deviation rate.

The numerical figures in this notebook (number of traces, crash
rate, classifier scores in notebooks `06` and `07`) are
therefore not intended to replicate any of the paper's reported
metrics. The reproducibility artefact for the paper's tables is
in notebooks `01`-`04`.


## Load the feature CSV

The CSV at `../data/picoc_features.csv` is produced by
`scripts/extract_features_from_waveforms.py` from the raw
`.npz` traces. The traces themselves are not redistributed in
this repository.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display

FEATURES_CSV = Path("..") / "data" / "picoc_features.csv"
FIGURES_DIR = Path("..") / "figures" / "em_evidence"

df = pd.read_csv(FEATURES_CSV)
df.head()


## Descriptive statistics


In [ ]:
print(f"Total rows: {len(df)}")
print()
print("Rows per campaign phase:")
print(df["campaign_phase"].value_counts().to_string())
print()
print("Anomalous vs normal:")
counts = df["is_anomalous"].value_counts()
for value, n in counts.items():
    label = "anomalous" if value else "normal"
    pct = 100.0 * n / len(df)
    print(f"  {label:<10} {n:>6}  ({pct:5.2f}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["duration"].dropna(), bins=60, color="#1f4e79", edgecolor="white")
axes[0].set_xlabel("Execution duration (s)")
axes[0].set_ylabel("Number of traces")
axes[0].set_title("Distribution of execution duration")
axes[0].grid(True, linestyle=":", alpha=0.5)

anomalous = df[df["is_anomalous"]]
if len(anomalous) > 0:
    exit_counts = anomalous["exit_code"].value_counts().sort_index()
    axes[1].bar(exit_counts.index.astype(str), exit_counts.values, color="#a23b3b")
    axes[1].set_xlabel("Exit code")
    axes[1].set_ylabel("Number of anomalous traces")
    axes[1].set_title("Exit-code distribution within anomalous traces")
    axes[1].grid(True, axis="y", linestyle=":", alpha=0.5)
else:
    axes[1].text(0.5, 0.5, "no anomalous traces", ha="center", va="center")

fig.tight_layout()
plt.show()


## Reference figure: baseline calibration profile

The figure below shows the baseline EM profile of the target
board in idle, with reference traces for CPU-bound,
Memory-bound, and IO-bound microbenchmarks. These centroids
define the subsystem-attribution criterion used in agreement
scoring.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_calibration_profile.pdf"), width=800, height=600))


## Reference figure: mean trace by exit code

Mean EM trace comparison emitted by the production capture
pipeline, contrasting traces with `exit_code == 0` against
traces with `exit_code != 0`. As above, the underlying split is
on the picoc exit code, not on EM deviation; this figure is
included to show that the capture pipeline produces a visible
signal difference between the two populations.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_normal_vs_anomalous.pdf"), width=800, height=600))


## Next steps

Notebook 06 projects the 21-dimensional feature space into
two dimensions via PCA and t-SNE, and shows the corresponding
reference figures from the EM analysis pipeline.

Notebook 07 covers the diagnostic outputs of the EM-side
classifier and reproduces a baseline detection-metric report.
